# Pattern 00: Setup and Orientation

This notebook verifies your environment is ready and explains the framework architecture.

## Prerequisites

1. Docker Compose running: `docker compose up -d` (starts Keycloak + OPA)
2. Python deps installed: `uv sync`
3. `.env` file with `OPENAI_API_KEY` set

## 1. Health checks

Verify that Keycloak and OPA are running and reachable.

In [1]:
import httpx
from framework.config import KEYCLOAK_URL, OPA_URL

# Keycloak health (management interface is on port 9000)
keycloak_health = KEYCLOAK_URL.replace(":8080", ":9000")
r = httpx.get(f"{keycloak_health}/health/ready", timeout=10)
print(f"Keycloak: {r.status_code} {r.json().get('status', 'unknown')}")

# OPA health
r = httpx.get(f"{OPA_URL}/health", timeout=5)
print(f"OPA: {r.status_code}")

Keycloak: 200 UP
OPA: 200


## 2. Verify Keycloak realm

Fetch a token for alice to confirm the realm is loaded and users exist.

In [2]:
from framework.auth_helpers import fetch_user_jwt, decode_jwt
from framework.display import show_token

token = fetch_user_jwt("alice")
show_token(token, label="alice's JWT")

                               alice's JWT                                
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ claim              ┃ value                                             ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ sub                │ 7f78567c-4e16-44d4-b723-dca7328abd88              │
│ preferred_username │ alice                                             │
│ aud                │ [document-service-client, expense-service-client] │
│ azp                │ agent-client                                      │
│ iss                │ http://localhost:8080/realms/agentauth            │
│ role               │ employee                                          │
│ department         │ engineering                                       │
│ reports_to         │ bob                                               │
│ scope              │ openid agent-claims                               │
│ exp                │ 1776174281                                        │
│ iat                │ 1776172481                                        │
│ jti                │ onrtro:c1815dfd-b62b-1fa4-bc1b-d4c87161a850       │
│ sid                │ Uyns_jCTFqLvw3Gyf7Z3xUZO                          │
│ typ                │ Bearer                                            │
└────────────────────┴───────────────────────────────────────────────────┘

## 3. Architecture overview

Each pattern follows the same request flow:

```
Agent --> Expense MCP Server (HTTP) --> Expense FastAPI Service
Agent --> Document MCP Server (HTTP) --> Document FastAPI Service
```

### Why MCP servers?

You don't strictly need MCP servers to demonstrate these auth patterns. The agent
could call the backend services directly. We include them for two reasons:

1. **Realism.** In production agent deployments, tools are typically exposed through
   MCP servers rather than called as raw HTTP endpoints. The MCP server is the
   integration point between the agent framework and the backend service.

2. **MCP has first-class OAuth2 support.** The MCP specification includes a full
   authorization framework: MCP servers can require OAuth2 Bearer tokens on the
   transport layer, advertise OAuth metadata endpoints, and validate tokens via
   introspection or JWKS. In this repo we keep the MCP servers as simple
   pass-through proxies to focus on the auth patterns themselves, but in production
   you would likely configure OAuth2 at the MCP transport level rather than
   handling it in application code.

   See the [MCP authorization spec](https://modelcontextprotocol.io/specification/2025-03-26/basic/authorization)
   for details on how MCP servers can enforce OAuth2 natively.

### The role of the MCP server in each pattern

In this repo, the MCP server is infrastructure. It forwards credentials from the
agent to the backend service. It never makes authorization decisions. The auth
logic lives in two small files per pattern:

- **`mcp_auth.py`**: what credentials the MCP server attaches to outbound requests
  (API key, user header, JWT, exchanged token)
- **`service_auth.py`**: how the backend service extracts identity and makes authz
  decisions from the inbound request

The progression is about what the **backend service** can do with whatever
identity information reaches it.

### The plugin interface

**MCP side** (what credentials to forward):

```python
class AuthHandler:
    async def prepare_request(self, user_context, headers) -> dict:
        """Add auth credentials to outbound request headers."""
        return headers
```

**Service side** (how to extract and validate identity):

```python
async def get_identity(request: Request) -> Identity:
    """Extract caller identity from the inbound request."""
```

### Why per-pattern isolation?

In production, you'd likely use a single flexible auth layer that supports multiple
methods. We deliberately didn't do that here. Each pattern has its own auth code, on
both the MCP server side and the service side, so you can read exactly what happens at
each boundary without tracing through abstractions. Once you understand each pattern
individually, consolidating them is straightforward.

## 4. Smoke test: start pattern 1 and run a query

Quick test that the full stack works: PatternRunner starts services + MCP servers,
the agent calls tools via MCP, and results come back.

In [3]:
from framework.runner import PatternRunner

runner = PatternRunner("p01_service_credential")
await runner.start()

[04/14/26 06:14:43] INFO     StreamableHTTP session manager started                  ]8;id=195597;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=681961;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py#128\128]8;;\

                    INFO     StreamableHTTP session manager started                  ]8;id=404668;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=546013;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py#128\128]8;;\

                    INFO     HTTP Request: POST http://127.0.0.1:64000/mcp "HTTP/1.1 200 OK"        ]8;id=49942;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=124693;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     Negotiated protocol version: 2025-11-25                         ]8;id=7800;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/client/streamable_http.py\streamable_http.py]8;;\:]8;id=834191;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/client/streamable_http.py#193\193]8;;\

                    INFO     Terminating session: None                                       ]8;id=71211;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http.py\streamable_http.py]8;;\:]8;id=46580;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http.py#785\785]8;;\

...

                    INFO     HTTP Request: POST http://127.0.0.1:64000/mcp "HTTP/1.1 200 OK"        ]8;id=309940;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=831778;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     HTTP Request: POST http://127.0.0.1:64001/mcp "HTTP/1.1 200 OK"        ]8;id=647016;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=517592;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     HTTP Request: POST http://127.0.0.1:64000/mcp "HTTP/1.1 200 OK"        ]8;id=185028;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=888209;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\

[04/14/26 06:15:05] INFO     StreamableHTTP session manager shutting down            ]8;id=376257;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=411661;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py#132\132]8;;\

                    INFO     StreamableHTTP session manager shutting down            ]8;id=666625;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=650191;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py#132\132]8;;\

In [4]:
await runner.run_as("alice", "What expenses do I have?")

                    INFO     HTTP Request: POST                                                     ]8;id=529301;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=572099;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:8080/realms/agentauth/protocol/openid-connect/token                  
                             "HTTP/1.1 200 OK"                                                                     

╭──────────────────────────────────╮
│ [alice] What expenses do I have? │
╰──────────────────────────────────╯

                    INFO     Terminating session: None                                       ]8;id=124597;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http.py\streamable_http.py]8;;\:]8;id=297661;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http.py#785\785]8;;\

                    INFO     Processing request of type ListToolsRequest                              ]8;id=302991;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=63765;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/lowlevel/server.py#727\727]8;;\

                    INFO     Terminating session: None                                       ]8;id=262123;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http.py\streamable_http.py]8;;\:]8;id=501272;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http.py#785\785]8;;\

...

[04/14/26 06:14:53] INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=241108;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=839051;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             204 No Content"                                                                       

[04/14/26 06:15:04] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=303466;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=582637;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             OK"                                                                                   

                                                    tool calls                                                     
┏━━━━━┳━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ tool         ┃ args ┃ status ┃ result                                                                     ┃
┡━━━━━╇━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ get_expenses │ {}   │      - │ {'type': 'text', 'text': '{"identity_method": "api_key",                   │
│     │              │      │        │ "identity_detail": "shared service credential, no user identity", "count": │
│     │              │      │        │ 8, "expenses": [{"id": 1, "user_id": "alice", "department": "engineering", │
│     │              │      │        │ "amount": 42.5, "category": "software", "description": "JetBrains AI       │
│     │              │      │        │ assistant subscription",                                                   │
└─────┴──────────────┴──────┴────────┴────────────────────────────────────────────────────────────────────────────┘

╭──────────────────────────────────────────────────── answer ─────────────────────────────────────────────────────╮
│ I can’t determine your identity in this session (uses a shared service credential), so I can’t pull your        │
│ personal expenses right now.                                                                                    │
│                                                                                                                 │
│ How would you like to proceed?                                                                                  │
│ - Provide your user_id (e.g., alice, bob) and I’ll fetch your expenses.                                         │
│ - Re-authenticate with your user account to allow me to view your expenses.                                     │
│ - If you want to see expenses by department instead, tell me the department name and I’ll list those.           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

AgentResult(content='I can’t determine your identity in this session (uses a shared service credential), so I can’t pull your personal expenses right now.\n\nHow would you like to proceed?\n- Provide your user_id (e.g., alice, bob) and I’ll fetch your expenses.\n- Re-authenticate with your user account to allow me to view your expenses.\n- If you want to see expenses by department instead, tell me the department name and I’ll list those.', tool_calls=[ToolCallTrace(name='get_expenses', args={}, status=None, result_summary='{\'type\': \'text\', \'text\': \'{"identity_method": "api_key", "identity_detail": "shared service credential, no user identity", "count": 8, "expenses": [{"id": 1, "user_id": "alice", "department": "engineering", "amount": 42.5, "category": "software", "description": "JetBrains AI assistant subscription",', error=None)])

In [5]:
await runner.show_service_identity()

                    INFO     HTTP Request: GET http://127.0.0.1:63998/debug/last-request "HTTP/1.1  ]8;id=576562;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=984721;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             200 OK"                                                                               

╭──────── expense-service /debug/last-request ─────────╮
│ method:  api_key                                     │
│ user_id: <none>                                      │
│ detail:  shared service credential, no user identity │
│                                                      │
╰──────────────────────────────────────────────────────╯

                    INFO     HTTP Request: GET http://127.0.0.1:63999/debug/last-request "HTTP/1.1  ]8;id=195829;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=67971;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             200 OK"                                                                               

╭─ document-service /debug/last-request ─╮
│ method:  never                         │
│ user_id: <none>                        │
│ detail:  no requests yet               │
│                                        │
╰────────────────────────────────────────╯

In [6]:
await runner.stop()

Pattern p01_service_credential stopped.

[04/14/26 06:15:08] INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=963667;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=122452;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             204 No Content"                                                                       

## Setup complete

Everything is working. Proceed to pattern 01 to start the progression.

Each pattern notebook follows the same structure:
1. `PatternRunner("pXX_name")` to wire up the pattern
2. `show_auth_code()` to read the auth logic (this IS the lesson)
3. `start()` to bring up the full stack
4. `run_as(user, prompt)` to see the pattern in action
5. `show_service_identity()` to see what the service actually received
6. `stop()` to clean up